# iWildCam – Metadata EDA

No images are loaded. All analysis is done from `metadata.csv` (~30 MB).

**Before running:** execute `python ../download_metadata.py` once to fetch the CSV.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent.parent))
from iwildcam.config import CONFIG

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)

META_PATH = Path(CONFIG['data_dir']) / 'iwildcam_v2.0' / 'metadata.csv'
print(f'Metadata path: {META_PATH}')
print(f'Exists: {META_PATH.exists()}')

## 1. Load metadata

In [ ]:
df = pd.read_csv(META_PATH)
print(f'Rows: {len(df):,}   Columns: {list(df.columns)}')
df.head(3)

## 2. Split overview

In [ ]:
rows = []
for split, grp in df.groupby('split'):
    rows.append({
        'split':            split,
        'samples':          len(grp),
        'unique_classes':   grp['y'].nunique(),
        'unique_locations': grp['location_remapped'].nunique(),
        'id_or_ood':        'OOD' if split in ('val', 'test') else 'ID',
    })

overview = pd.DataFrame(rows).set_index('split')
print(overview.to_string())

## 3. Class imbalance (training split)

In [ ]:
train = df[df['split'] == 'train']
class_counts = train['y'].value_counts().sort_values(ascending=False)

print(f'Training samples       : {len(train):,}')
print(f'Classes in train       : {len(class_counts)}')
print(f'Most common class      : {class_counts.iloc[0]:,} images')
print(f'Median class size      : {int(class_counts.median()):,} images')
print(f'Rarest class           : {class_counts.iloc[-1]:,} images')
print(f'Imbalance ratio        : {class_counts.iloc[0] / class_counts.iloc[-1]:.0f}x')

tiny = (class_counts < 10).sum()
print(f'Classes with <10 imgs  : {tiny} ({100*tiny/len(class_counts):.0f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(len(class_counts)), class_counts.values, width=1.0, color='steelblue')
axes[0].set_yscale('log')
axes[0].set_xlabel('Class rank')
axes[0].set_ylabel('Images (log scale)')
axes[0].set_title('Class frequency — sorted (full dataset)')

axes[1].hist(class_counts.values, bins=40, color='steelblue', edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xlabel('Images per class')
axes[1].set_ylabel('Number of classes (log)')
axes[1].set_title('Distribution of class sizes (full dataset)')

plt.tight_layout()
plt.savefig(RESULTS / 'class_imbalance_full.png', bbox_inches='tight')
plt.show()

## 4. Domain (location) distribution

In [ ]:
loc_counts = train['location_remapped'].value_counts().sort_values(ascending=False)

print(f'Training locations     : {len(loc_counts)}')
print(f'Max images / location  : {loc_counts.iloc[0]:,}')
print(f'Median images / loc    : {int(loc_counts.median()):,}')
print(f'Min images / location  : {loc_counts.iloc[-1]:,}')

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(loc_counts)), loc_counts.values, width=1.0, color='darkorange')
ax.set_yscale('log')
ax.set_xlabel('Location rank')
ax.set_ylabel('Images (log scale)')
ax.set_title('Images per camera location — training set (full dataset)')
plt.tight_layout()
plt.savefig(RESULTS / 'location_distribution_full.png', bbox_inches='tight')
plt.show()

## 5. ID vs OOD domain overlap

In [ ]:
def locs(split_name):
    return set(df[df['split'] == split_name]['location_remapped'].unique())

train_locs = locs('train')
print('Overlap with training locations:')
for name, key in [('id_val', 'id_val'), ('ood_val', 'val'),
                  ('id_test', 'id_test'), ('ood_test', 'test')]:
    s = locs(key)
    print(f'  {name:10s}: {len(s):3d} locations | overlap with train: {len(s & train_locs)}')

## 6. Class overlap across splits

In [ ]:
def classes(split_name):
    return set(df[df['split'] == split_name]['y'].unique())

train_cls = classes('train')
print('Class overlap with training set:')
for name, key in [('id_val', 'id_val'), ('ood_val', 'val'),
                  ('id_test', 'id_test'), ('ood_test', 'test')]:
    s = classes(key)
    print(f'  {name:10s}: {len(s):3d} classes | shared: {len(s & train_cls)} | only in split: {len(s - train_cls)}')

## 7. Species richness per location

In [ ]:
richness = train.groupby('location_remapped')['y'].nunique()

print(f'Avg species / location : {richness.mean():.1f}')
print(f'Max species / location : {richness.max()}')
print(f'Min species / location : {richness.min()}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(richness.values, bins=30, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Distinct species at location')
ax.set_ylabel('Number of locations')
ax.set_title('Species richness per camera location (train)')
plt.tight_layout()
plt.savefig(RESULTS / 'species_richness_full.png', bbox_inches='tight')
plt.show()

## 8. Temporal distribution

In [ ]:
df['datetime_parsed'] = pd.to_datetime(df['datetime'], errors='coerce')
df['month'] = df['datetime_parsed'].dt.month
df['hour']  = df['datetime_parsed'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split_label, split_key, color in [
    ('Train (ID)', 'train', 'steelblue'),
    ('OOD test',   'test',  'tomato'),
]:
    sub = df[df['split'] == split_key]
    axes[0].hist(sub['month'].dropna(), bins=12, range=(1,13), alpha=0.6,
                 label=split_label, color=color, density=True)
    axes[1].hist(sub['hour'].dropna(),  bins=24, range=(0,24), alpha=0.6,
                 label=split_label, color=color, density=True)

for ax, xlabel, title in zip(axes,
    ['Month', 'Hour of day'],
    ['Capture month distribution', 'Capture hour distribution']):
    ax.set_xlabel(xlabel); ax.set_ylabel('Density')
    ax.set_title(title);   ax.legend()

plt.tight_layout()
plt.savefig(RESULTS / 'temporal_distribution.png', bbox_inches='tight')
plt.show()